# CTMatch Evaluation Baseline

Establishes baseline metrics on TREC 2021 and KZ datasets using the existing pipeline.
This is the regression gate — all subsequent changes must match or beat these numbers.

**Filters tested:**
- `sim` — embedding + category similarity (MiniLM-L6-v2 + bart-large-mnli)
- `svm` — linear SVM on embeddings
- `classifier` — fine-tuned SciBERT
- Individual and combined filter configurations

**Metrics:** Mean First Positive Rank (FPR), Mean Reciprocal Rank (MRR/FRR), Mean F1

In [ ]:
# Option 1: Install from local clone (if you cloned the repo in Colab)
# !git clone https://github.com/semajyllek/ctmatch.git
# import sys; sys.path.insert(0, "/content/ctmatch/src")

# Option 2: If running locally alongside the repo
import sys; sys.path.insert(0, "../src")

# Option 3: After pushing changes to GitHub
# !pip install -q git+https://github.com/semajyllek/ctmatch.git

# Dependencies
# !pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml


In [ ]:
# Upload eval data files or mount Google Drive
# These files are in ctmatch/data/ locally
import os

# Option 1: Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/ct_data23/evaluation'

# Option 2: Local
# DATA_ROOT = os.environ.get('CTMATCH_DATA_ROOT', '../data')

TREC_REL_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_judgments.txt')
KZ_REL_PATH = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')
TREC_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_topics.xml')
KZ_TOPIC_PATH = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')

for p in [TREC_REL_PATH, KZ_REL_PATH, TREC_TOPIC_PATH, KZ_TOPIC_PATH]:
    print(f"{chr(10003) if os.path.exists(p) else chr(10007)} {p}")

In [ ]:
from ctmatch.evaluation.evaluator import EvaluatorConfig, Evaluator

## Run baseline: SVM + Classifier (no gen, no OpenAI needed)

In [ ]:
eval_config = EvaluatorConfig(
    rel_paths=[TREC_REL_PATH, KZ_REL_PATH],
    trec_topic_path=TREC_TOPIC_PATH,
    kz_topic_path=KZ_TOPIC_PATH,
    max_topics=50,
    filters=['svm', 'classifier'],
)

ev = Evaluator(eval_config)
results_svm_clf = ev.evaluate()
print('SVM + Classifier results:')
for k, v in results_svm_clf.items():
    print(f'  {k}: {v:.4f}')

## Run baseline: SIM + SVM + Classifier

In [ ]:
eval_config_full = EvaluatorConfig(
    rel_paths=[TREC_REL_PATH, KZ_REL_PATH],
    trec_topic_path=TREC_TOPIC_PATH,
    kz_topic_path=KZ_TOPIC_PATH,
    max_topics=50,
    filters=['sim', 'svm', 'classifier'],
)

ev_full = Evaluator(eval_config_full)
results_sim_svm_clf = ev_full.evaluate()
print('SIM + SVM + Classifier results:')
for k, v in results_sim_svm_clf.items():
    print(f'  {k}: {v:.4f}')

## Run individual filters for comparison

In [ ]:
all_results = {}

for filter_name in ['sim', 'svm', 'classifier']:
    print(f'\n--- Running {filter_name} only ---')
    cfg = EvaluatorConfig(
        rel_paths=[TREC_REL_PATH, KZ_REL_PATH],
        trec_topic_path=TREC_TOPIC_PATH,
        kz_topic_path=KZ_TOPIC_PATH,
        max_topics=50,
        filters=[filter_name],
    )
    ev_single = Evaluator(cfg)
    res = ev_single.evaluate()
    all_results[filter_name] = res
    for k, v in res.items():
        print(f'  {k}: {v:.4f}')

## Detailed evaluation with per-example predictions

Saves every (topic, doc, predicted, actual) to JSONL for error analysis.

In [ ]:
eval_config_detailed = EvaluatorConfig(
    rel_paths=[TREC_REL_PATH, KZ_REL_PATH],
    trec_topic_path=TREC_TOPIC_PATH,
    kz_topic_path=KZ_TOPIC_PATH,
    max_topics=50,
    filters=["svm", "classifier"],
)

ev_detailed = Evaluator(eval_config_detailed)
detailed_results = ev_detailed.evaluate_detailed(output_path="eval_predictions.jsonl")
print(f"Saved {detailed_results['total_examples']} predictions ({detailed_results['total_errors']} errors)")
print(f"Output: {detailed_results['output_path']}")
for k, v in detailed_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

In [ ]:
# Preview the predictions file
import json
with open("eval_predictions.jsonl") as f:
    lines = f.readlines()
print(f"{len(lines)} total predictions")
errors = [json.loads(l) for l in lines if json.loads(l)["is_error"]]
print(f"{len(errors)} errors")
print("
First 3 errors:")
for ex in errors[:3]:
    print(f"  topic={ex['topic_id']} doc={ex['doc_id']} pred={ex['predicted_label']} actual={ex['actual_label']}")
    print(f"    topic: {ex['topic_text'][:100]}...")
    print(f"    doc:   {ex['doc_text'][:100]}...")
    print()

In [ ]:
# Download the predictions file (Colab)
# from google.colab import files
# files.download("eval_predictions.jsonl")

# Or copy the content to paste into Claude Code:
# with open("eval_predictions.jsonl") as f:
#     print(f.read())

## Summary table

In [ ]:
import pandas as pd

all_results['svm+classifier'] = results_svm_clf
all_results['sim+svm+classifier'] = results_sim_svm_clf

df = pd.DataFrame(all_results).T[['mean_fpr', 'mean_frr', 'mean_f1']]
df.columns = ['Mean FPR (lower=better)', 'Mean MRR (higher=better)', 'Mean F1 (higher=better)']
df.round(4)

## Previous results reference

From the original `ctmatch_ir_demo_and_eval.ipynb` with gen filter (OpenAI text-davinci-003):
- `mean_fpr: 7.0`
- `mean_frr (MRR): 0.218`
- `mean_f1: 0.224`

These used `filters=['gen']` only on 30 topics.